In [2]:
import os

print(os.getcwd())
PROJECT_PATH = r"D:\My_projects\User Frustration Prediction - Github"

d:\My_projects\User Frustration Prediction - Github


In [2]:
import pandas as pd

df = pd.read_csv(PROJECT_PATH + r"\data\AllProductReviews.csv")
print(df.columns.tolist())

df.head()


['ReviewTitle', 'ReviewBody', 'ReviewStar', 'Product']


,ReviewTitle,ReviewBody,ReviewStar,Product
0,Honest review of an edm music lover\n,No doubt it has a great bass and to a great ex...,3,boAt Rockerz 255
1,Unreliable earphones with high cost\n,"This earphones are unreliable, i bought it be...",1,boAt Rockerz 255
2,Really good and durable.\n,"i bought itfor 999,I purchased it second time,...",4,boAt Rockerz 255
3,stopped working in just 14 days\n,Its sound quality is adorable. overall it was ...,1,boAt Rockerz 255
4,Just Awesome Wireless Headphone under 1000...😉\n,Its Awesome... Good sound quality & 8-9 hrs ba...,5,boAt Rockerz 255


In [7]:
df['ReviewBody'].head()

0    No doubt it has a great bass and to a great ex...
1    This  earphones are unreliable, i bought it be...
2    i bought itfor 999,I purchased it second time,...
3    Its sound quality is adorable. overall it was ...
4    Its Awesome... Good sound quality & 8-9 hrs ba...
Name: ReviewBody, dtype: object

In [4]:
import re

frustration_keywords = [
    "waste", "useless", "not working", "stopped", "disappointed", "poor", "frustrating",
    "doesn't work", "irritating", "worst", "problem", "issue", "disconnect", "bad", "low quality",
    "not worthy", "not worth", "unworthy", "stops", "noisy", "crash", "lag", "uncomfortable", "unreliable",
    "not reliable", "cracked", "crack", "isn't working"
]

negation_words = {"not","no","never","n't","without","hardly","barely"}

def label_frustration(text,window=3):
    text = str(text).lower()
    words = re.findall(r"\b\w+\b",text)

    for kw in frustration_keywords:
        kw_words = kw.split()
        kw_len = len(kw_words)

        for i in range(len(words) - kw_len + 1):
            if words[i:i + kw_len] == kw_words:
                start = max(0, i-window)
                preceding = words[start:i]
                if not any(neg in " ".join(preceding) for neg in negation_words):
                    return 1

    return 0

df['Frustration_Label'] = df['ReviewBody'].apply(label_frustration)
df = df[['ReviewBody','Frustration_Label','Product']].dropna()
df = df.rename(columns={"ReviewBody":"text", "Frustration_Label": "label"})
print(df['label'].value_counts())

label
0    11393
1     2944
Name: count, dtype: int64


In [5]:
TEST_PRODUCT = "boAt Rockerz 255"

train_df = df[df['Product'] != TEST_PRODUCT].drop(columns='Product').reset_index(drop=True)
test_df = df[df['Product'] == TEST_PRODUCT].drop(columns ='Product').reset_index(drop=True)


print('Train shape:',train_df.shape)
print('Test shape:', test_df.shape)
print("\nTrain label distribution:\n", train_df['label'].value_counts())
print("\nTest label distribution:\n", test_df['label'].value_counts())

train_df.to_csv(PROJECT_PATH + r"\train.csv", index = False)
test_df.to_csv(PROJECT_PATH + r"\test.csv", index = False)

Train shape: (9337, 2)
Test shape: (5000, 2)

Train label distribution:
 label
0    7594
1    1743
Name: count, dtype: int64

Test label distribution:
 label
0    3799
1    1201
Name: count, dtype: int64


In [6]:
print(df['Product'].nunique())
print(df.groupby('Product')['label'].mean().sort_values())

10
Product
Flybot Boom              0.073171
Flybot Beat              0.086957
PTron Intunes            0.098425
Sennheiser CX 6.0BT      0.144200
Flybot Wave              0.148352
Samsung EO-BG950CBEIN    0.178571
JBL T205BT               0.220411
boAt Rockerz 255         0.240200
Skullcandy S2PGHW-174    0.255230
JBL T110BT               0.286160
Name: label, dtype: float64


In [8]:
from sklearn.model_selection import LeaveOneGroupOut
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score

logo = LeaveOneGroupOut()
groups = df['Product'].values
X_text = df['text'].astype(str).values
y = df['label'].values

results = []
for train_idx, test_idx in logo.split(X_text, y, groups):
    held_out = df['Product'].iloc[test_idx].unique()[0]
    vec = TfidfVectorizer(max_features=5000,ngram_range=(1,2))
    X_train_fold = vec.fit_transform(X_text[train_idx])
    X_test_fold = vec.transform(X_text[test_idx])
    clf = LogisticRegression(max_iter = 1000, class_weight = 'balanced')
    clf.fit(X_train_fold, y[train_idx])
    preds = clf.predict(X_test_fold)

    results.append({
        'held_out_product': held_out,
        'test_size': len(test_idx),
        'accuracy': accuracy_score(y[test_idx],preds),
        'f1': f1_score(y[test_idx],preds)
    })

results_df = pd.DataFrame(results).sort_values('test_size', ascending = False)
print(results_df)
print("\nWeighted avg accuracy:", (results_df['accuracy'] * results_df['test_size']).sum() / results_df['test_size'].sum())

        held_out_product  test_size  accuracy        f1
7    Sennheiser CX 6.0BT       5000  0.956800  0.855034
9       boAt Rockerz 255       5000  0.945200  0.890662
3             JBL T110BT       1922  0.959417  0.931818
4             JBL T205BT       1411  0.964564  0.921384
5          PTron Intunes        254  0.948819  0.779661
8  Skullcandy S2PGHW-174        239  0.953975  0.916031
6  Samsung EO-BG950CBEIN        224  0.977679  0.939759
2            Flybot Wave        182  0.978022  0.925926
1            Flybot Boom         82  0.951220  0.714286
0            Flybot Beat         23  1.000000  1.000000

Weighted avg accuracy: 0.9543140126944271


In [9]:
import string

def clean_text(text):
    text = str(text).lower()
    text = re.sub(r"\n", " ", text)
    text = re.sub(r"http\S+|www\S+", "", text)
    text = text.translate(str.maketrans("", "", string.punctuation))
    text = re.sub(r"\s+", " ", text).strip()
    return text

train_df["clean_text"] = train_df["text"].apply(clean_text)
test_df["clean_text"] = test_df["text"].apply(clean_text)
print(train_df[["text", "clean_text", "label"]].head())

                                                text  \
0  Its a cheap chinese product. No bass at all. V...   
1  SOUND QUALITY of flybot wave wireless earphone...   
2                              It's China product.\n   
3  Its an awesome purchase but the mike doesn't q...   
4  When put to full volume..sound wasn’t clear en...   

                                          clean_text  label  
0  its a cheap chinese product no bass at all vol...      0  
1  sound quality of flybot wave wireless earphone...      1  
2                                  its china product      0  
3  its an awesome purchase but the mike doesnt qu...      0  
4  when put to full volumesound wasn’t clear enou...      0  


In [10]:
tfidf = TfidfVectorizer(max_features=5000, ngram_range=(1, 2), stop_words="english")

X_train = tfidf.fit_transform(train_df["clean_text"])
X_test = tfidf.transform(test_df["clean_text"])
y_train = train_df["label"]
y_test = test_df["label"]

print("X_train shape:", X_train.shape)
print("X_test shape:", X_test.shape)

X_train shape: (9337, 5000)
X_test shape: (5000, 5000)


In [11]:
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.svm import LinearSVC
import joblib

logreg = LogisticRegression(max_iter=1000, class_weight="balanced")
logreg.fit(X_train, y_train)
logreg_pred = logreg.predict(X_test)
print("LogReg Accuracy:", accuracy_score(y_test, logreg_pred))
print(classification_report(y_test, logreg_pred))

svm_model = LinearSVC(class_weight="balanced", random_state=42)
svm_model.fit(X_train, y_train)
svm_pred = svm_model.predict(X_test)
print("SVM Accuracy:", accuracy_score(y_test, svm_pred))
print(classification_report(y_test, svm_pred))

joblib.dump(svm_model, "linear_svm_model.pkl")
joblib.dump(tfidf, "tfidf_vectorizer.pkl")

LogReg Accuracy: 0.936
              precision    recall  f1-score   support

           0       0.97      0.95      0.96      3799
           1       0.85      0.90      0.87      1201

    accuracy                           0.94      5000
   macro avg       0.91      0.92      0.91      5000
weighted avg       0.94      0.94      0.94      5000

SVM Accuracy: 0.9316
              precision    recall  f1-score   support

           0       0.96      0.95      0.95      3799
           1       0.85      0.87      0.86      1201

    accuracy                           0.93      5000
   macro avg       0.90      0.91      0.91      5000
weighted avg       0.93      0.93      0.93      5000



['tfidf_vectorizer.pkl']

In [12]:
import torch
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from torch.optim import AdamW
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

model_name = "prajjwal1/bert-mini"
tokenizer = AutoTokenizer.from_pretrained(model_name)
bert_model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=2,
    id2label={0: "Not Frustrated", 1: "Frustrated"},
    label2id={"Not Frustrated": 0, "Frustrated": 1}
)

c:\Users\Adhar\anaconda3\envs\tf_env\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
c:\Users\Adhar\anaconda3\envs\tf_env\lib\site-packages\huggingface_hub\file_download.py:943: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
W0930 19:40:38.921000 128512 site-packages\torch\distributed\elastic\multiprocessing\redirects.py:29] NOTE: Redirects are currently not supported in Windows or MacOs.
Some weights of BertForSequenceClassification were not initialized from the model checkpoint at prajjwal1/bert-mini and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predict

In [13]:
class ReviewDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_length=128):
        self.texts = texts.tolist()
        self.labels = labels.tolist()
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, index):
        encoding = self.tokenizer(
            self.texts[index], truncation=True, padding="max_length",
            max_length=self.max_length, return_tensors="pt"
        )
        return {
            "input_ids": encoding["input_ids"].squeeze(0),
            "attention_mask": encoding["attention_mask"].squeeze(0),
            "labels": torch.tensor(self.labels[index], dtype=torch.long)
        }

In [14]:
train_dataset = ReviewDataset(train_df["text"], train_df["label"], tokenizer)
test_dataset = ReviewDataset(test_df["text"], test_df["label"], tokenizer)

train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=16, shuffle=False)

In [15]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

bert_model = bert_model.to(device)
optimizer = AdamW(bert_model.parameters(), lr=2e-5)
epochs = 2

Using device: cpu


In [16]:
bert_model.train()
for epoch in range(epochs):
    total_loss = 0
    for batch in train_loader:
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["labels"].to(device)

        optimizer.zero_grad()
        outputs = bert_model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
        loss = outputs.loss
        total_loss += loss.item()
        loss.backward()
        optimizer.step()

    print(f"Epoch {epoch + 1}/{epochs}, Training Loss: {total_loss / len(train_loader):.4f}")

Epoch 1/2, Training Loss: 0.2761
Epoch 2/2, Training Loss: 0.0978


In [17]:
bert_model.eval()
all_predictions, all_labels = [], []

with torch.no_grad():
    for batch in test_loader:
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["labels"].to(device)
        outputs = bert_model(input_ids=input_ids, attention_mask=attention_mask)
        predictions = torch.argmax(outputs.logits, dim=1)
        all_predictions.extend(predictions.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

print("Accuracy:", accuracy_score(all_labels, all_predictions))
print(classification_report(all_labels, all_predictions, digits=4))
print(confusion_matrix(all_labels, all_predictions))

Accuracy: 0.9608
              precision    recall  f1-score   support

           0     0.9876    0.9605    0.9738      3799
           1     0.8851    0.9617    0.9218      1201

    accuracy                         0.9608      5000
   macro avg     0.9363    0.9611    0.9478      5000
weighted avg     0.9629    0.9608    0.9613      5000

[[3649  150]
 [  46 1155]]


In [18]:
NEW_MODEL_DIR = "bert_mini_frustration_model_v2_productsplit"
bert_model.save_pretrained(NEW_MODEL_DIR)
tokenizer.save_pretrained(NEW_MODEL_DIR)
print(f"Saved to {NEW_MODEL_DIR} — verify before replacing saved_model")

Saved to bert_mini_frustration_model_v2_productsplit — verify before replacing saved_model


In [19]:
def predict_with_bert(review):
    bert_model.eval()
    encoding = tokenizer(review, truncation=True, padding="max_length", max_length=128, return_tensors="pt")
    input_ids = encoding["input_ids"].to(device)
    attention_mask = encoding["attention_mask"].to(device)
    with torch.no_grad():
        outputs = bert_model(input_ids=input_ids, attention_mask=attention_mask)
    prediction = torch.argmax(outputs.logits, dim=1).item()
    return "Frustrated" if prediction == 1 else "Not Frustrated"

print(predict_with_bert("The product stopped working after two days and customer support is useless."))
print(predict_with_bert("Excellent product, amazing quality and very good performance."))

Frustrated
Not Frustrated


In [23]:
from transformers import TextClassificationPipeline
import numpy as np

pipe = TextClassificationPipeline(model=bert_model.to("cpu"), tokenizer=tokenizer, top_k=None)

def get_frustrated_confidence(text):
    scores = pipe(str(text)[:512])[0]
    for s in scores:
        if s['label'] == 'Frustrated':
            return s['score']
    return None

eval_df = test_df.copy()
eval_df['model_confidence'] = eval_df['text'].apply(get_frustrated_confidence)
eval_df['review_id'] = range(len(eval_df))

high_conf_frustrated = eval_df[eval_df['model_confidence'] >= 0.9]
high_conf_not_frustrated = eval_df[eval_df['model_confidence'] <= 0.1]
borderline = eval_df[(eval_df['model_confidence'] > 0.4) & (eval_df['model_confidence'] < 0.6)]

n_per_bucket = 60
sample = pd.concat([
    high_conf_frustrated.sample(min(n_per_bucket, len(high_conf_frustrated)), random_state=42),
    high_conf_not_frustrated.sample(min(n_per_bucket, len(high_conf_not_frustrated)), random_state=42),
    borderline.sample(min(n_per_bucket, len(borderline)), random_state=42)
]).sample(frac=1, random_state=1).reset_index(drop=True)

blind_df = sample[['review_id', 'text']].copy()
blind_df['your_manual_label'] = ''
blind_df.to_csv(PROJECT_PATH + r"\manual_labeling_TO_FILL.csv", index=False)

answer_key = sample[['review_id', 'text', 'model_confidence']].copy()
answer_key.to_csv(PROJECT_PATH + r"\manual_labeling_ANSWER_KEY_dont_open_yet.csv", index=False)

print(f"Buckets: {len(high_conf_frustrated)} high-conf-frustrated (sampled {min(n_per_bucket,len(high_conf_frustrated))}), "
      f"{len(high_conf_not_frustrated)} high-conf-not (sampled {min(n_per_bucket,len(high_conf_not_frustrated))}), "
      f"{len(borderline)} borderline (sampled {min(n_per_bucket,len(borderline))})")

Buckets: 1176 high-conf-frustrated (sampled 60), 3495 high-conf-not (sampled 60), 54 borderline (sampled 54)


In [24]:
print("test_df shape:", test_df.shape)
print("eval_df shape:", eval_df.shape)
print("model_confidence nulls:", eval_df['model_confidence'].isnull().sum())
print("high_conf_frustrated:", len(high_conf_frustrated))
print("high_conf_not_frustrated:", len(high_conf_not_frustrated))
print("borderline:", len(borderline))

test_df shape: (5000, 3)
eval_df shape: (5000, 5)
model_confidence nulls: 0
high_conf_frustrated: 1176
high_conf_not_frustrated: 3495
borderline: 54


In [3]:
import pandas as pd
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, classification_report

PROJECT_PATH = r"D:\My_projects\User Frustration Prediction - Github"

manual_df = pd.read_csv(PROJECT_PATH + r"\manual_labeling_TO_FILL.csv")
answer_key = pd.read_csv(PROJECT_PATH + r"\manual_labeling_ANSWER_KEY_dont_open_yet.csv")

merged = manual_df.merge(answer_key, on = ['review_id','text'], how = 'inner')
assert len(merged) == len(manual_df), "Row count mismatch after merge - check review_id alignment"

merged['model_prediction'] = (merged['model_confidence'] >= 0.5).astype(int)

y_true = merged['your_manual_label']
y_pred = merged['model_prediction']

print("REAL accuracy vs manual ground truth:", accuracy_score(y_true, y_pred))
print("Precision:", precision_score(y_true, y_pred))
print("Recall:", recall_score(y_true, y_pred))
print("F1:", f1_score(y_true, y_pred))
print()
print("Confidence matrix:\n", confusion_matrix(y_true, y_pred))
print()
print(classification_report(y_true, y_pred, target_names=["Not Frustrated", "Frustrated"]))

high_conf_mask = (merged['model_confidence'] >= 0.9) | (merged['model_confidence'] <= 0.1)
high_conf_subset = merged[high_conf_mask]
print(f"\nAccuracy on HIGH-CONFIDENCE predictions only (n={len(high_conf_subset)}):",
      accuracy_score(high_conf_subset['your_manual_label'],
                      (high_conf_subset['model_confidence'] >= 0.5).astype(int)))

borderline_mask = (merged['model_confidence'] > 0.4) & (merged['model_confidence'] < 0.6)
borderline_subset = merged[borderline_mask]
print(f"Accuracy on BORDERLINE predictions only (n={len(borderline_subset)}):",
      accuracy_score(borderline_subset['your_manual_label'],
                      (borderline_subset['model_confidence'] >= 0.5).astype(int)))

merged.to_csv(PROJECT_PATH + r"\manual_validation_results.csv", index=False)

REAL accuracy vs manual ground truth: 0.7413793103448276
Precision: 0.7469879518072289
Recall: 0.7209302325581395
F1: 0.7337278106508875

Confidence matrix:
 [[67 21]
 [24 62]]

                precision    recall  f1-score   support

Not Frustrated       0.74      0.76      0.75        88
    Frustrated       0.75      0.72      0.73        86

      accuracy                           0.74       174
     macro avg       0.74      0.74      0.74       174
  weighted avg       0.74      0.74      0.74       174


Accuracy on HIGH-CONFIDENCE predictions only (n=120): 0.7916666666666666
Accuracy on BORDERLINE predictions only (n=54): 0.6296296296296297
